# DataJam 2026 · ConviveData  
## Cuaderno 02 · Ejecución de indicadores y pruebas estadísticas

Este cuaderno construye los insumos del visor territorial a partir de las bases limpias y ejecuta pruebas estadísticas exploratorias.

Puede ejecutarse en Colab leyendo los archivos desde GitHub Raw o localmente si las carpetas del repositorio ya existen.


## 1. Librerías

In [ ]:
from pathlib import Path
import urllib.request
import warnings

import numpy as np
import pandas as pd

try:
    from scipy.stats import pearsonr, spearmanr, kendalltau
    SCIPY_DISPONIBLE = True
except Exception:
    SCIPY_DISPONIBLE = False
    print("Advertencia: scipy no está disponible. No se calcularán pruebas estadísticas.")

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)

## 2. Configuración reproducible de rutas

In [ ]:
REPO_RAW = "https://raw.githubusercontent.com/00Jake00/convivedata-vif-bogota-2018-2025-Datajam/main"

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = BASE_DIR / "outputs_limpieza"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ARCHIVOS_LIMPIOS = {
    "vif": {
        "local": OUTPUT_DIR / "vif_limpia_localizada_2018_2025.csv",
        "url": f"{REPO_RAW}/outputs_limpieza/vif_limpia_localizada_2018_2025.csv",
    },
    "lesiones": {
        "local": OUTPUT_DIR / "lesiones_limpia_localizada_2018_2025.csv",
        "url": f"{REPO_RAW}/outputs_limpieza/lesiones_limpia_localizada_2018_2025.csv",
    },
    "spa": {
        "local": OUTPUT_DIR / "spa_abusivo_limpio_localizado_2018_2025.csv",
        "url": f"{REPO_RAW}/outputs_limpieza/spa_abusivo_limpio_localizado_2018_2025.csv",
    },
    "resumen_limpieza": {
        "local": OUTPUT_DIR / "resumen_limpieza_fuentes.csv",
        "url": f"{REPO_RAW}/outputs_limpieza/resumen_limpieza_fuentes.csv",
    },
}

def descargar_si_no_existe(nombre):
    ruta = ARCHIVOS_LIMPIOS[nombre]["local"]
    url = ARCHIVOS_LIMPIOS[nombre]["url"]

    if not ruta.exists():
        print(f"Descargando {nombre} desde GitHub Raw...")
        urllib.request.urlretrieve(url, ruta)
    else:
        print(f"{nombre}: archivo local encontrado")

    return ruta

def leer_csv_seguro(ruta):
    for encoding in ["utf-8-sig", "utf-8", "latin1", "cp1252"]:
        try:
            return pd.read_csv(ruta, encoding=encoding)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"No se pudo leer el archivo: {ruta}")

print("BASE_DIR:", BASE_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("PROCESSED_DIR:", PROCESSED_DIR)

## 3. Carga de bases limpias

In [ ]:
ruta_vif = descargar_si_no_existe("vif")
ruta_lesiones = descargar_si_no_existe("lesiones")
ruta_spa = descargar_si_no_existe("spa")
ruta_resumen_limpieza = descargar_si_no_existe("resumen_limpieza")

vif = leer_csv_seguro(ruta_vif)
lesiones = leer_csv_seguro(ruta_lesiones)
spa_abusivo = leer_csv_seguro(ruta_spa)
resumen_limpieza = leer_csv_seguro(ruta_resumen_limpieza)

print("VIF:", vif.shape)
print("Lesiones:", lesiones.shape)
print("SPA abusivo:", spa_abusivo.shape)
print("Resumen limpieza:", resumen_limpieza.shape)

## 4. Controles mínimos de estructura

In [ ]:
def validar_columnas(df, columnas, nombre):
    faltantes = [c for c in columnas if c not in df.columns]
    if faltantes:
        raise ValueError(f"{nombre}: faltan columnas requeridas: {faltantes}")
    print(f"{nombre}: columnas requeridas completas")

validar_columnas(vif, ["ANIO", "loc_norm", "localidad_valida", "CANTIDAD", "SEXO", "RANGO_VITAL"], "VIF")
validar_columnas(lesiones, ["ANIO", "loc_norm", "localidad_valida", "CANTIDAD"], "Lesiones")
validar_columnas(spa_abusivo, ["ANO", "loc_norm", "localidad_valida", "CASOS"], "SPA abusivo")

## 5. Población de referencia

In [ ]:
poblacion_referencia = pd.DataFrame([
    ("USAQUEN", "Usaquén", 501999),
    ("CHAPINERO", "Chapinero", 139701),
    ("SANTA FE", "Santa Fe", 110048),
    ("SAN CRISTOBAL", "San Cristóbal", 404697),
    ("USME", "Usme", 457302),
    ("TUNJUELITO", "Tunjuelito", 199430),
    ("BOSA", "Bosa", 673077),
    ("KENNEDY", "Kennedy", 1088443),
    ("FONTIBON", "Fontibón", 394648),
    ("ENGATIVA", "Engativá", 887080),
    ("SUBA", "Suba", 1218513),
    ("BARRIOS UNIDOS", "Barrios Unidos", 243465),
    ("TEUSAQUILLO", "Teusaquillo", 153025),
    ("LOS MARTIRES", "Los Mártires", 99119),
    ("ANTONIO NARINO", "Antonio Nariño", 109176),
    ("PUENTE ARANDA", "Puente Aranda", 258287),
    ("LA CANDELARIA", "La Candelaria", 24088),
    ("RAFAEL URIBE URIBE", "Rafael Uribe Uribe", 374246),
    ("CIUDAD BOLIVAR", "Ciudad Bolívar", 707569),
    ("SUMAPAZ", "Sumapaz", 6531),
], columns=["loc_norm", "localidad", "poblacion_ref"])

poblacion_referencia

## 6. Filtro temporal y registros localizados

In [ ]:
vif = vif[(vif["ANIO"].between(2018, 2025)) & (vif["localidad_valida"] == True)].copy()
lesiones = lesiones[(lesiones["ANIO"].between(2018, 2025)) & (lesiones["localidad_valida"] == True)].copy()
spa_abusivo = spa_abusivo[(spa_abusivo["ANO"].between(2018, 2025)) & (spa_abusivo["localidad_valida"] == True)].copy()

print("Registros después de control 2018-2025 y localidad válida")
print("VIF:", len(vif), "| casos:", int(vif["CANTIDAD"].sum()))
print("Lesiones:", len(lesiones), "| casos:", int(lesiones["CANTIDAD"].sum()))
print("SPA abusivo:", len(spa_abusivo), "| casos:", int(spa_abusivo["CASOS"].sum()))

## 7. Agregación por localidad

In [ ]:
vif_casos = (
    vif.groupby("loc_norm", as_index=False)["CANTIDAD"]
    .sum()
    .rename(columns={"CANTIDAD": "vif_casos_2018_2025"})
)

lesiones_casos = (
    lesiones.groupby("loc_norm", as_index=False)["CANTIDAD"]
    .sum()
    .rename(columns={"CANTIDAD": "lesiones_casos_2018_2025"})
)

spa_casos = (
    spa_abusivo.groupby("loc_norm", as_index=False)["CASOS"]
    .sum()
    .rename(columns={"CASOS": "spa_abusivo_casos_2018_2025"})
)

vif_casos.sort_values("vif_casos_2018_2025", ascending=False).head(10)

## 8. Persistencia anual de VIF

In [ ]:
vif_anual = (
    vif.groupby(["loc_norm", "ANIO"], as_index=False)["CANTIDAD"]
    .sum()
    .merge(poblacion_referencia[["loc_norm", "poblacion_ref"]], on="loc_norm", how="left")
)

vif_anual["tasa_anual_100k"] = vif_anual["CANTIDAD"] / vif_anual["poblacion_ref"] * 100000

mediana_anual = (
    vif_anual.groupby("ANIO", as_index=False)["tasa_anual_100k"]
    .median()
    .rename(columns={"tasa_anual_100k": "mediana_anual_100k"})
)

vif_anual = vif_anual.merge(mediana_anual, on="ANIO", how="left")
vif_anual["sobre_mediana"] = vif_anual["tasa_anual_100k"] > vif_anual["mediana_anual_100k"]

persistencia_vif = (
    vif_anual.groupby("loc_norm", as_index=False)["sobre_mediana"]
    .sum()
    .rename(columns={"sobre_mediana": "vif_persistencia_anios"})
)

persistencia_vif.sort_values("vif_persistencia_anios", ascending=False).head(10)

## 9. Participación de mujeres, niñas, niños y adolescentes

In [ ]:
vif["flag_mujeres_nna"] = (
    vif["SEXO"].astype(str).str.upper().eq("FEMENINO") |
    vif["RANGO_VITAL"].astype(str).str.upper().isin(["INFANCIA", "ADOLESCENCIA"])
)

num_mujeres_nna = vif.loc[vif["flag_mujeres_nna"]].groupby("loc_norm")["CANTIDAD"].sum()
den_vif = vif.groupby("loc_norm")["CANTIDAD"].sum()

pct_mujeres_nna = (
    (num_mujeres_nna / den_vif * 100)
    .round(2)
    .rename("pct_mujeres_nna_vif")
    .reset_index()
)

pct_mujeres_nna.sort_values("pct_mujeres_nna_vif", ascending=False).head(10)

## 10. Base integrada por localidad

In [ ]:
base_resumen = (
    poblacion_referencia
    .merge(vif_casos, on="loc_norm", how="left")
    .merge(persistencia_vif, on="loc_norm", how="left")
    .merge(pct_mujeres_nna, on="loc_norm", how="left")
    .merge(lesiones_casos, on="loc_norm", how="left")
    .merge(spa_casos, on="loc_norm", how="left")
)

for col in ["vif_casos_2018_2025", "vif_persistencia_anios", "lesiones_casos_2018_2025", "spa_abusivo_casos_2018_2025"]:
    base_resumen[col] = base_resumen[col].fillna(0).astype(int)

base_resumen["vif_tasa_prom_anual_100k"] = (
    base_resumen["vif_casos_2018_2025"] / base_resumen["poblacion_ref"] * 100000 / 8
).round(2)

base_resumen["lesiones_tasa_prom_anual_100k"] = (
    base_resumen["lesiones_casos_2018_2025"] / base_resumen["poblacion_ref"] * 100000 / 8
).round(2)

base_resumen["spa_abusivo_tasa_prom_anual_100k"] = (
    base_resumen["spa_abusivo_casos_2018_2025"] / base_resumen["poblacion_ref"] * 100000 / 8
).round(2)

base_resumen.head()

## 11. Índices de lectura territorial

In [ ]:
def minmax_100(serie):
    serie = pd.Series(serie, dtype=float)
    minimo = serie.min()
    maximo = serie.max()
    if maximo == minimo:
        return pd.Series(0, index=serie.index, dtype=float)
    return (serie - minimo) / (maximo - minimo) * 100

base_resumen["indice_vif_familiar"] = (
    0.40 * minmax_100(base_resumen["vif_tasa_prom_anual_100k"]) +
    0.30 * minmax_100(base_resumen["vif_persistencia_anios"]) +
    0.20 * minmax_100(base_resumen["pct_mujeres_nna_vif"]) +
    0.10 * minmax_100(base_resumen["vif_casos_2018_2025"])
).round(2)

base_resumen["indice_contexto"] = (
    0.50 * minmax_100(base_resumen["lesiones_tasa_prom_anual_100k"]) +
    0.50 * minmax_100(base_resumen["spa_abusivo_tasa_prom_anual_100k"])
).round(2)

base_resumen["brecha_vif_contexto"] = (
    base_resumen["indice_vif_familiar"] - base_resumen["indice_contexto"]
).round(2)

base_resumen[["localidad", "indice_vif_familiar", "indice_contexto", "brecha_vif_contexto"]].sort_values("indice_vif_familiar", ascending=False).head(10)

## 12. Rankings y lectura operativa

In [ ]:
base_resumen["rank_vif_carga"] = base_resumen["vif_casos_2018_2025"].rank(ascending=False, method="min").astype(int)
base_resumen["rank_vif_tasa"] = base_resumen["vif_tasa_prom_anual_100k"].rank(ascending=False, method="min").astype(int)
base_resumen["rank_vif_familiar"] = base_resumen["indice_vif_familiar"].rank(ascending=False, method="min").astype(int)
base_resumen["cambio_carga_vs_tasa"] = base_resumen["rank_vif_carga"] - base_resumen["rank_vif_tasa"]

def nivel_indice(valor):
    if valor >= 66:
        return "alto"
    if valor >= 33:
        return "medio"
    return "bajo"

base_resumen["nivel_vif_familiar"] = base_resumen["indice_vif_familiar"].apply(nivel_indice)
base_resumen["nivel_contexto"] = base_resumen["indice_contexto"].apply(nivel_indice)

def tipo_decision(row):
    if row["nivel_vif_familiar"] == "alto" and row["nivel_contexto"] == "alto":
        return "VIF alta con entorno complejo"
    if row["nivel_vif_familiar"] == "alto":
        return "VIF alta con contexto moderado"
    if row["nivel_vif_familiar"] == "medio":
        return "Seguimiento focalizado"
    return "Monitoreo general"

def alerta_lectura(row):
    if row["poblacion_ref"] < 50000:
        return "Alta: población pequeña. Interpretar tasas con cautela."
    if row["localidad"] in ["Los Mártires", "Santa Fe", "Chapinero"]:
        return "Media: posible efecto de población flotante o centralidad urbana."
    return "Baja: lectura territorial más estable."

def accion_sugerida(row):
    if row["nivel_vif_familiar"] == "alto" and row["nivel_contexto"] == "alto":
        return "Conviene revisar la oferta de atención familiar junto con salud mental, convivencia y presencia institucional en el territorio."
    if row["nivel_vif_familiar"] == "alto":
        return "La prioridad debería estar en rutas de atención, prevención en hogares, enfoque de género y protección de niñas, niños y adolescentes."
    if row["brecha_vif_contexto"] > 8:
        return "La VIF pesa más que las señales de contexto. Vale la pena mirar con más detalle la dinámica familiar y las rutas de atención."
    return "Mantener seguimiento y contrastar con nuevos cortes de información y conocimiento territorial."

base_resumen["tipo_decision"] = base_resumen.apply(tipo_decision, axis=1)
base_resumen["alerta_lectura"] = base_resumen.apply(alerta_lectura, axis=1)
base_resumen["accion_sugerida"] = base_resumen.apply(accion_sugerida, axis=1)

## 13. Texto breve para ficha territorial

In [ ]:
def formato_miles(valor):
    return f"{int(valor):,}".replace(",", ".")

def etiqueta_carga(rank):
    if rank <= 5:
        return "alta"
    if rank <= 15:
        return "media"
    return "baja"

def construir_lectura(row):
    texto = (
        f"Registra {formato_miles(row['vif_casos_2018_2025'])} casos entre 2018 y 2025, "
        f"con una carga {etiqueta_carga(row['rank_vif_carga'])} frente al resto de localidades. "
        f"La tasa promedio es de {row['vif_tasa_prom_anual_100k']:.0f} por 100.000 habitantes. "
        f"El índice VIF es {row['nivel_vif_familiar']} y el contexto territorial es {row['nivel_contexto']}."
    )
    if row["alerta_lectura"].startswith("Alta"):
        texto += " La tasa debe leerse con cautela porque la población es pequeña."
    elif row["alerta_lectura"].startswith("Media"):
        texto += " Conviene revisar población flotante y dinámica urbana antes de sacar conclusiones."
    return texto

base_resumen["lectura_breve"] = base_resumen.apply(construir_lectura, axis=1)
base_resumen["lectura_breve_final"] = base_resumen["lectura_breve"]

columnas_finales = [
    "localidad", "loc_norm", "poblacion_ref",
    "vif_casos_2018_2025", "vif_tasa_prom_anual_100k", "vif_persistencia_anios", "pct_mujeres_nna_vif",
    "lesiones_casos_2018_2025", "lesiones_tasa_prom_anual_100k",
    "spa_abusivo_casos_2018_2025", "spa_abusivo_tasa_prom_anual_100k",
    "indice_vif_familiar", "indice_contexto", "brecha_vif_contexto",
    "rank_vif_carga", "rank_vif_tasa", "rank_vif_familiar", "cambio_carga_vs_tasa",
    "nivel_vif_familiar", "nivel_contexto", "tipo_decision",
    "alerta_lectura", "accion_sugerida", "lectura_breve", "lectura_breve_final",
]

base_resumen = base_resumen[columnas_finales].sort_values("rank_vif_familiar").reset_index(drop=True)
base_resumen.head(10)

## 14. Base localidad-año

In [ ]:
vif_anio = (
    vif.groupby(["loc_norm", "ANIO"], as_index=False)["CANTIDAD"]
    .sum()
    .rename(columns={"ANIO": "anio", "CANTIDAD": "vif_casos"})
)

lesiones_anio = (
    lesiones.groupby(["loc_norm", "ANIO"], as_index=False)["CANTIDAD"]
    .sum()
    .rename(columns={"ANIO": "anio", "CANTIDAD": "lesiones_casos"})
)

spa_anio = (
    spa_abusivo.groupby(["loc_norm", "ANO"], as_index=False)["CASOS"]
    .sum()
    .rename(columns={"ANO": "anio", "CASOS": "spa_abusivo_casos"})
)

# Base completa de 20 localidades x 8 años.
base_localidad_anio = (
    poblacion_referencia.assign(key=1)
    .merge(pd.DataFrame({"anio": list(range(2018, 2026)), "key": 1}), on="key")
    .drop(columns="key")
    .merge(vif_anio, on=["loc_norm", "anio"], how="left")
    .merge(lesiones_anio, on=["loc_norm", "anio"], how="left")
    .merge(spa_anio, on=["loc_norm", "anio"], how="left")
)

for col in ["vif_casos", "lesiones_casos", "spa_abusivo_casos"]:
    base_localidad_anio[col] = base_localidad_anio[col].fillna(0).astype(int)

base_localidad_anio["vif_tasa_100k"] = (
    base_localidad_anio["vif_casos"] / base_localidad_anio["poblacion_ref"] * 100000
).round(2)

base_localidad_anio["lesiones_tasa_100k"] = (
    base_localidad_anio["lesiones_casos"] / base_localidad_anio["poblacion_ref"] * 100000
).round(2)

base_localidad_anio["spa_abusivo_tasa_100k"] = (
    base_localidad_anio["spa_abusivo_casos"] / base_localidad_anio["poblacion_ref"] * 100000
).round(2)

base_localidad_anio.head()

## 15. Validaciones de calidad

In [ ]:
validaciones = pd.DataFrame([
    {
        "validacion": "Localidades en base resumen",
        "valor_observado": int(base_resumen["loc_norm"].nunique()),
        "valor_esperado": 20,
        "estado": "OK" if base_resumen["loc_norm"].nunique() == 20 else "REVISAR",
    },
    {
        "validacion": "Observaciones localidad-año",
        "valor_observado": int(len(base_localidad_anio)),
        "valor_esperado": 160,
        "estado": "OK" if len(base_localidad_anio) == 160 else "REVISAR",
    },
    {
        "validacion": "Casos VIF localizados",
        "valor_observado": int(base_resumen["vif_casos_2018_2025"].sum()),
        "valor_esperado": 283773,
        "estado": "OK" if int(base_resumen["vif_casos_2018_2025"].sum()) == 283773 else "REVISAR",
    },
    {
        "validacion": "Casos lesiones localizados",
        "valor_observado": int(base_resumen["lesiones_casos_2018_2025"].sum()),
        "valor_esperado": 162475,
        "estado": "OK" if int(base_resumen["lesiones_casos_2018_2025"].sum()) == 162475 else "REVISAR",
    },
    {
        "validacion": "Casos SPA abusivo localizados",
        "valor_observado": int(base_resumen["spa_abusivo_casos_2018_2025"].sum()),
        "valor_esperado": 74238,
        "estado": "OK" if int(base_resumen["spa_abusivo_casos_2018_2025"].sum()) == 74238 else "REVISAR",
    },
    {
        "validacion": "Duplicados por localidad",
        "valor_observado": int(base_resumen["loc_norm"].duplicated().sum()),
        "valor_esperado": 0,
        "estado": "OK" if int(base_resumen["loc_norm"].duplicated().sum()) == 0 else "REVISAR",
    },
])

validaciones

## 16. Pruebas estadísticas exploratorias

In [ ]:
def prueba_asociacion(df, x, y, nivel, relacion, nota):
    datos = df[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    fila = {
        "nivel": nivel,
        "relacion": relacion,
        "n": len(datos),
        "pearson_r": np.nan,
        "pearson_p": np.nan,
        "spearman_rho": np.nan,
        "spearman_p": np.nan,
        "kendall_tau": np.nan,
        "kendall_p": np.nan,
        "nota": nota,
    }

    if SCIPY_DISPONIBLE and len(datos) >= 3:
        pearson = pearsonr(datos[x], datos[y])
        spearman = spearmanr(datos[x], datos[y])
        kendall = kendalltau(datos[x], datos[y])

        fila.update({
            "pearson_r": pearson.statistic,
            "pearson_p": pearson.pvalue,
            "spearman_rho": spearman.statistic,
            "spearman_p": spearman.pvalue,
            "kendall_tau": kendall.statistic,
            "kendall_p": kendall.pvalue,
        })

    return fila


pruebas = []

# Nivel localidad acumulado 2018-2025.
pruebas.extend([
    prueba_asociacion(base_resumen, "vif_casos_2018_2025", "lesiones_casos_2018_2025",
                      "localidad_total_2018_2025", "VIF casos vs lesiones casos",
                      "20 localidades, agregadas 2018-2025"),
    prueba_asociacion(base_resumen, "vif_casos_2018_2025", "spa_abusivo_casos_2018_2025",
                      "localidad_total_2018_2025", "VIF casos vs SPA casos",
                      "20 localidades, agregadas 2018-2025"),
    prueba_asociacion(base_resumen, "lesiones_casos_2018_2025", "spa_abusivo_casos_2018_2025",
                      "localidad_total_2018_2025", "Lesiones casos vs SPA casos",
                      "20 localidades, agregadas 2018-2025"),
    prueba_asociacion(base_resumen, "vif_tasa_prom_anual_100k", "lesiones_tasa_prom_anual_100k",
                      "localidad_total_2018_2025", "VIF tasa vs lesiones tasa",
                      "20 localidades, agregadas 2018-2025"),
    prueba_asociacion(base_resumen, "vif_tasa_prom_anual_100k", "spa_abusivo_tasa_prom_anual_100k",
                      "localidad_total_2018_2025", "VIF tasa vs SPA tasa",
                      "20 localidades, agregadas 2018-2025"),
    prueba_asociacion(base_resumen, "lesiones_tasa_prom_anual_100k", "spa_abusivo_tasa_prom_anual_100k",
                      "localidad_total_2018_2025", "Lesiones tasa vs SPA tasa",
                      "20 localidades, agregadas 2018-2025"),
])

# Nivel localidad-año.
pruebas.extend([
    prueba_asociacion(base_localidad_anio, "vif_casos", "lesiones_casos",
                      "localidad_anio_2018_2025", "VIF casos vs lesiones casos",
                      "160 observaciones localidad-año"),
    prueba_asociacion(base_localidad_anio, "vif_casos", "spa_abusivo_casos",
                      "localidad_anio_2018_2025", "VIF casos vs SPA casos",
                      "160 observaciones localidad-año"),
    prueba_asociacion(base_localidad_anio, "lesiones_casos", "spa_abusivo_casos",
                      "localidad_anio_2018_2025", "Lesiones casos vs SPA casos",
                      "160 observaciones localidad-año"),
    prueba_asociacion(base_localidad_anio, "vif_tasa_100k", "lesiones_tasa_100k",
                      "localidad_anio_2018_2025", "VIF tasa vs lesiones tasa",
                      "160 observaciones localidad-año"),
    prueba_asociacion(base_localidad_anio, "vif_tasa_100k", "spa_abusivo_tasa_100k",
                      "localidad_anio_2018_2025", "VIF tasa vs SPA tasa",
                      "160 observaciones localidad-año"),
    prueba_asociacion(base_localidad_anio, "lesiones_tasa_100k", "spa_abusivo_tasa_100k",
                      "localidad_anio_2018_2025", "Lesiones tasa vs SPA tasa",
                      "160 observaciones localidad-año"),
])

# Sensibilidad sin localidades centrales con posible efecto de población flotante.
excluir = ["LOS MARTIRES", "SANTA FE", "LA CANDELARIA"]
base_sensibilidad = base_resumen[~base_resumen["loc_norm"].isin(excluir)].copy()

pruebas.extend([
    prueba_asociacion(base_sensibilidad, "vif_tasa_prom_anual_100k", "lesiones_tasa_prom_anual_100k",
                      "sin_los_martires_santa_fe_la_candelaria", "VIF tasa vs lesiones tasa",
                      "Sensibilidad excluyendo localidades centrales con población pequeña/flotante"),
    prueba_asociacion(base_sensibilidad, "vif_tasa_prom_anual_100k", "spa_abusivo_tasa_prom_anual_100k",
                      "sin_los_martires_santa_fe_la_candelaria", "VIF tasa vs SPA tasa",
                      "Sensibilidad excluyendo localidades centrales con población pequeña/flotante"),
])

resultados_pruebas = pd.DataFrame(pruebas)

columnas_num = ["pearson_r", "pearson_p", "spearman_rho", "spearman_p", "kendall_tau", "kendall_p"]
for col in columnas_num:
    resultados_pruebas[col] = pd.to_numeric(resultados_pruebas[col], errors="coerce")

resultados_pruebas

## 17. Spearman por año

In [ ]:
filas_spearman = []

for anio, grupo in base_localidad_anio.groupby("anio"):
    comparaciones = [
        ("VIF tasa vs lesiones tasa", "vif_tasa_100k", "lesiones_tasa_100k"),
        ("VIF tasa vs SPA tasa", "vif_tasa_100k", "spa_abusivo_tasa_100k"),
        ("Lesiones tasa vs SPA tasa", "lesiones_tasa_100k", "spa_abusivo_tasa_100k"),
    ]

    for relacion, x, y in comparaciones:
        datos = grupo[[x, y]].dropna()
        rho, p = (np.nan, np.nan)

        if SCIPY_DISPONIBLE and len(datos) >= 3:
            r = spearmanr(datos[x], datos[y])
            rho, p = r.statistic, r.pvalue

        filas_spearman.append({
            "anio": anio,
            "relacion": relacion,
            "n": len(datos),
            "spearman_rho": rho,
            "spearman_p": p,
        })

spearman_por_anio = pd.DataFrame(filas_spearman)
spearman_por_anio

## 18. Resumen de fuentes

In [ ]:
resumen_fuentes = pd.DataFrame([
    {
        "fuente": "Violencia intrafamiliar",
        "registros_limpios": len(vif),
        "casos_2018_2025_20_localidades": int(base_resumen["vif_casos_2018_2025"].sum()),
        "uso": "Fenómeno principal",
    },
    {
        "fuente": "Lesiones personales",
        "registros_limpios": len(lesiones),
        "casos_2018_2025_20_localidades": int(base_resumen["lesiones_casos_2018_2025"].sum()),
        "uso": "Señal de contexto territorial",
    },
    {
        "fuente": "Consumo abusivo SPA",
        "registros_limpios": len(spa_abusivo),
        "casos_2018_2025_20_localidades": int(base_resumen["spa_abusivo_casos_2018_2025"].sum()),
        "uso": "Señal de contexto territorial",
    },
])

resumen_fuentes

## 19. Exportación de resultados

In [ ]:
salidas = {
    "base_resumen_localidad_2018_2025.csv": base_resumen,
    "base_localidad_anio_2018_2025.csv": base_localidad_anio,
    "resultados_pruebas_estadisticas.csv": resultados_pruebas,
    "spearman_por_anio.csv": spearman_por_anio,
    "resumen_fuentes.csv": resumen_fuentes,
    "validaciones_calidad.csv": validaciones,
}

for nombre, df in salidas.items():
    df.to_csv(OUTPUT_DIR / nombre, index=False, encoding="utf-8-sig")
    df.to_csv(PROCESSED_DIR / nombre, index=False, encoding="utf-8-sig")

print("Archivos exportados en:", OUTPUT_DIR)
for archivo in sorted(OUTPUT_DIR.glob("*.csv")):
    print("-", archivo.name)

print("\nCopia adicional en:", PROCESSED_DIR)

## 20. Resumen final

In [ ]:
print("Resumen de salida")
print("Localidades:", base_resumen["loc_norm"].nunique())
print("Observaciones localidad-año:", len(base_localidad_anio))
print("Total VIF:", int(base_resumen["vif_casos_2018_2025"].sum()))
print("Total lesiones:", int(base_resumen["lesiones_casos_2018_2025"].sum()))
print("Total SPA abusivo:", int(base_resumen["spa_abusivo_casos_2018_2025"].sum()))
print("Mayor índice VIF:", base_resumen.sort_values("indice_vif_familiar", ascending=False).iloc[0]["localidad"])
print("Mayor brecha VIF-contexto:", base_resumen.sort_values("brecha_vif_contexto", ascending=False).iloc[0]["localidad"])

display(base_resumen[[
    "localidad", "vif_casos_2018_2025", "vif_tasa_prom_anual_100k",
    "indice_vif_familiar", "indice_contexto", "brecha_vif_contexto",
    "tipo_decision"
]].head(10))

display(resultados_pruebas)